# Lab 8: Working with Relational Data: Trees and Networks

## Objectives

By the end of this lab, you should be able to:

- Explain what makes data hierarchical, and how that differs from the flat, tabular shape used everywhere else this semester
- Reshape grouped, aggregated data into the structure tree charts expect
- Build and interpret a treemap and a sunburst chart, and choose between them for a given hierarchy
- Use tree charts' built-in click-to-drill-down interaction as a continuation of Lab 7's interactive EDA skills
- Explain how network data differs from hierarchical data, and represent a network with `networkx`
- Compare network layout algorithms and choose one that supports the point you want to make
- Turn a `networkx` layout into an interactive Plotly network chart

## Setup

We'll use a retail sales dataset for this lab: individual orders from a superstore chain, with each order tagged by region, product category, and product sub-category.

In [ ]:
# !pip install pandas plotly

import pandas as pd
import plotly.express as px

sales = pd.read_csv("superstore.csv")
sales.head()

,region,category,sub_category,segment,sales,profit,quantity,order_date
0,Nunavut,Office Supplies,Storage & Organization,Small Business,261.5400,-213.25,6,10/13/2010
1,Nunavut,Office Supplies,Appliances,Consumer,10123.0200,457.81,49,10/1/2012
2,Nunavut,Office Supplies,Binders and Binder Accessories,Consumer,244.5700,46.71,27,10/1/2012
3,Nunavut,Technology,Telephones and Communication,Corporate,4965.7595,1198.97,30,7/10/2011
4,Nunavut,Office Supplies,Appliances,Consumer,394.2700,30.94,19,8/28/2010


## 1. What makes data hierarchical?

Every dataset we've used so far this semester has been flat: rows and columns, with no notion that one category contains another. Hierarchical data is different. It has **levels of nesting**, where each level belongs entirely inside the level above it.

Some familiar examples:

- A **file system**: a folder contains files and other folders, which contain more files and folders.
- An **org chart**: a company contains divisions, which contain teams, which contain individual employees.
- A **taxonomy**: a kingdom contains phyla, which contain classes, which contain species.

Our dataset today has a hierarchy too, even though it arrived as a flat table: every order belongs to a **region**, within that region to a **product category**, and within that category to a **sub-category**. `region -> category -> sub_category` is a three-level hierarchy hiding inside a flat CSV, and building a tree chart means making that structure explicit.

## 2. Preparing hierarchical data

The plotting functions in this lab expect a "path plus value" structure: one or more columns describing the nesting path, and one numeric column describing the size of each leaf. This is the same aggregation skill from Lab 2, just aimed at a new destination shape instead of a wide table.

Let's start with the simplest version: total sales by region.

In [ ]:
# TODO: group the sales data by region and calculate the total sales for each region
region_sales = sales.groupby("region", as_index=False).agg(total_sales=("sales", "sum"))
region_sales.sort_values("total_sales", ascending=False)

,region,total_sales
6,West,3.597549e+06
3,Ontario,3.063212e+06
4,Prarie,2.837305e+06
0,Atlantic,2.014248e+06
5,Quebec,1.510195e+06
7,Yukon,9.758674e+05
1,Northwest Territories,8.008473e+05
2,Nunavut,1.163765e+05


That's a familiar one-level summary. Now let's extend it to two levels, region and category, which is where the hierarchy starts to matter.

In [ ]:
# TODO: group the sales data by region and category, and calculate the total sales for each combination
region_category_sales = (
    sales
    .groupby(["region", "category"], as_index=False)
    .agg(total_sales=("sales", "sum"))
)
region_category_sales.head(10)

,region,category,total_sales
0,Atlantic,Furniture,7.087268e+05
1,Atlantic,Office Supplies,4.784644e+05
2,Atlantic,Technology,8.270570e+05
3,Northwest Territories,Furniture,2.828379e+05
4,Northwest Territories,Office Supplies,2.168489e+05
5,Northwest Territories,Technology,3.011605e+05
6,Nunavut,Furniture,4.354561e+04
7,Nunavut,Office Supplies,3.861547e+04
8,Nunavut,Technology,3.421540e+04
9,Ontario,Furniture,1.109618e+06


This table already has everything a tree chart needs: a path (`region`, then `category`) and a value (`total_sales`). The only thing left is to hand it to a chart type built to show that nesting visually instead of as rows.

## 3. Treemap: nested rectangles

A **treemap** represents each item in the hierarchy as a rectangle, with area encoding the value. Rectangles are nested inside their parent's rectangle, so the whole chart is one large rectangle subdivided down through every level of the hierarchy.

In [ ]:
# TODO: create a treemap of total sales by region and category, with the size of each rectangle representing the total sales
fig = px.treemap(
    region_category_sales,
    path=["region", "category"],
    width=800,
    height=600,
    values="total_sales",
    title="Total sales by region and category"
)
fig.show()

The `path` argument is doing the real work here: it tells Plotly which columns describe the nesting, from outermost to innermost. Larger rectangles mean more sales; you can compare rectangles both within a region (which category dominates there) and across regions (which region is largest overall) in a single glance, something that would take several separate bar charts to show as clearly.

Let's extend to the full three-level hierarchy by adding `sub_category`.

In [ ]:
#TODO: add a third level to the treemap by grouping the sales data by region, category, and sub-category
full_hierarchy = (
    sales
    .groupby(["region", "category", "sub_category"], as_index=False)
    .agg(total_sales=("sales", "sum"))
)

#TODO: Update treemap to include the sub-category level, with the size of each rectangle representing the total sales
fig = px.treemap(
    full_hierarchy,
    path=["region", "category", "sub_category"],
    values="total_sales",
    color="category",
    title="Total sales by region, category, and sub-category"
)
fig.show()

`color="category"` here encodes the same variable already shown by position in the hierarchy; that's a deliberate redundancy, called redundant encoding, and it makes it much easier to visually track which sub-categories belong to which category once there are many small rectangles on screen.

**Note:** If the data being plotted contains negative numbers, your chart will be unable to render because Plotly cannot biuld a mark with negative area for those rows. Always check here if your figure doesn't render.

## 4. Sunburst: the same hierarchy, radially

A **sunburst chart** shows the identical kind of data, path plus value, but arranges it as rings instead of nested rectangles: the center ring is the outermost level of the hierarchy, and each ring further out represents one level deeper.

In [ ]:
fig = px.sunburst(
    full_hierarchy,
    path=["region", "category", "sub_category"],
    values="total_sales",
    color="category",
    title="Total sales by region, category, and sub-category (sunburst)"
)
fig.show()

**Choosing between them:**

- **Treemap**: area is easier to compare accurately than the wedge shapes a sunburst uses, especially for leaves that are far apart in the hierarchy. Better when your main question is "which items are the biggest," across the whole dataset.
- **Sunburst**: the ring structure makes the *depth* of the hierarchy and the parent-child relationships more immediately visible, since every level gets its own visually distinct ring. Better when your main question is about structure itself, how many levels deep does this go, and how does one level relate to the level above it, rather than precise size comparison.

Look at both charts above. Which one makes it easier to answer "which single sub-category has the highest sales, out of everything"? Which makes it easier to answer "how many categories does the West region have, and how do they compare to each other specifically within that region"?

## 5. Built-in interactivity: click to drill down

Both chart types support the same interaction, continuing directly from Lab 7: **click any segment to zoom into that branch**, and click the chart's title area (or use the path breadcrumb Plotly adds automatically) to zoom back out. No extra code is required for this; it comes built into both `px.treemap()` and `px.sunburst()`.

In [ ]:
fig = px.treemap(
    full_hierarchy,
    path=["region", "category", "sub_category"],
    values="total_sales",
    color="category",
    title="Click a region or category to drill in; click the breadcrumb to zoom back out"
)
fig.show()

Try clicking into a single region, then a single category within it. Notice the rectangles resize to fill the available space at each zoomed-in level, since the chart is now only showing that branch of the hierarchy. This kind of drill-down is often the fastest way to explore a hierarchy you don't know well yet: start at the top level, then follow whatever looks interesting deeper in, without writing a new filter for every step.


## 6. Beyond trees: network visualization

A hierarchy is one specific kind of relational structure: every item has **exactly one parent**. Many real datasets don't follow that rule. In a social network anyone can be connected to anyone else; in a flight network an airport connects to many others in every direction. Structures like these are **networks** (also called **graphs**), made of two things:

- **Nodes**: the items themselves (people, airports, web pages)
- **Edges**: the connections between pairs of nodes (friendships, flights, links)

In Python, the `networkx` library is the standard tool for building and analyzing networks. It also ships with several small, classic datasets, so we don't need a new file.

We'll use **Zachary's karate club**: 34 members of a university karate club, with an edge between two members if they were friends outside the club. During the study the club split into two factions, one following the instructor ("Mr. Hi") and one following the club president ("Officer"). Each node stores which faction that member joined in an attribute called `club`.

In [ ]:
# !pip install networkx matplotlib scipy

import networkx as nx
import matplotlib.pyplot as plt

G = nx.karate_club_graph()

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Each node carries attributes, just like a row in a table carries columns
G.nodes[0]

Notice how different this structure is from everything else in the lab: there's no `path` and no `value` column, just a list of who is connected to whom. Let's draw it with `networkx`'s built-in plotting function.

In [ ]:
# TODO: draw the network using networkx's default settings
nx.draw(G, with_labels=True)
plt.show()

### Layouts: where do the nodes go?

Here is the most important difference between network charts and the charts we've made so far. In a scatter plot, a point's x and y position *means something*: it encodes a data value. In a treemap, a rectangle's area encodes a value. In a network diagram, **node positions don't come from the data at all**. A **layout algorithm** decides where to put each node, and choosing that algorithm is a design decision, just like choosing a chart type.

`networkx` provides several layout functions. Each one returns a dictionary mapping every node to an `(x, y)` position:

- **Spring** (`spring_layout`): treats edges as springs and nodes as repelling magnets, then lets the system settle. Connected nodes end up close together, so clusters tend to emerge. This is the default used by `nx.draw()`.
- **Kamada-Kawai** (`kamada_kawai_layout`): tries to make the distance on screen between every pair of nodes match how many steps apart they are in the network. Often tidier than spring for small networks.
- **Circular** (`circular_layout`): places every node evenly around a circle. Very orderly, but it ignores the network's structure entirely, so clusters disappear.
- **Shell** (`shell_layout`): concentric circles, where *you* decide which nodes go in which ring. Useful when you already have a grouping you want to show.
- **Spectral** (`spectral_layout`): uses linear algebra on the network's connection matrix to place nodes. Good at revealing the network's largest divisions, but can bunch nodes on top of each other.
- **Random** (`random_layout`): exactly what it sounds like. Included here as a baseline for what a layout with *no* design intent looks like.

Let's compute all six and draw them side by side, coloring each node by its faction.

In [ ]:
club_colors = {"Mr. Hi": "#1f77b4", "Officer": "#ff7f0e"}
node_colors = [club_colors[G.nodes[n]["club"]] for n in G.nodes]

# Shell layout needs to be told which nodes belong in each ring: here, one ring per faction
factions = [[n for n in G.nodes if G.nodes[n]["club"] == club] for club in club_colors]

# TODO: compute a position dictionary for each layout
# (seed= makes the spring and random layouts come out the same every time you run the cell)
layouts = {
    "Spring": nx.spring_layout(G, seed=42),
    "Kamada-Kawai": nx.kamada_kawai_layout(G),
    "Circular": nx.circular_layout(G),
    "Shell": nx.shell_layout(G, nlist=factions),
    "Spectral": nx.spectral_layout(G),
    "Random": nx.random_layout(G, seed=42),
}

# TODO: draw the same network once per layout, in a 2 x 3 grid
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (name, pos) in zip(axes.flat, layouts.items()):
    nx.draw(G, pos, ax=ax, node_color=node_colors, node_size=150,
            edge_color="lightgray", with_labels=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

These six pictures contain **exactly the same data**: the same 34 nodes and the same 78 edges. Yet they tell very different stories:

- Spring makes the two factions visible as two clusters, with a few members bridging the gap. Kamada-Kawai hints at the same split, but less cleanly: the factions overlap in the middle.
- Circular is neat, but the clusters vanish into a tangle of crossing edges.
- Shell shows the two factions clearly, but only because *we* told it the factions in advance, so it can't help you discover a grouping you didn't already know about.
- Random is a reminder that a network drawing with no thought behind its layout communicates almost nothing.

Because the layout isn't data, readers can easily over-interpret it: two nodes drawn close together in a spring layout are *probably* well connected, but their exact distance isn't a measurement. When you present a network, choose the layout that supports the specific point you're making, and say so.

### Making the network interactive with Plotly

`nx.draw()` produces a static image. Since a layout is just a dictionary of `(x, y)` positions, we can hand those positions to Plotly instead and get the same interactivity we've used all semester: hover tooltips, zoom, and pan.

A Plotly network chart is built from two kinds of traces:

1. **One line trace for all the edges.** Each edge is a line from one node's position to another's. Putting `None` between edges tells Plotly to lift the pen, so they don't all join into one long line.
2. **A marker trace for the nodes.** We'll make one per faction, so each gets its own color and legend entry.

In [ ]:
import plotly.graph_objects as go


def network_xy(G, pos):
    """Return x and y lists for the edge trace, then one node trace per faction."""
    edge_x, edge_y = [], []
    for u, v in G.edges():
        edge_x += [pos[u][0], pos[v][0], None]  # None breaks the line between edges
        edge_y += [pos[u][1], pos[v][1], None]
    xs, ys = [edge_x], [edge_y]
    for members in factions:
        xs.append([pos[n][0] for n in members])
        ys.append([pos[n][1] for n in members])
    return xs, ys


def plot_network(G, pos, title):
    xs, ys = network_xy(G, pos)
    fig = go.Figure()
    # TODO: add one line trace for all of the edges
    fig.add_trace(go.Scatter(
        x=xs[0], y=ys[0], mode="lines",
        line=dict(color="lightgray", width=1),
        hoverinfo="skip", showlegend=False,
    ))
    # TODO: add one marker trace per faction, with a hover label for each member
    for i, (club, members) in enumerate(zip(club_colors, factions), start=1):
        fig.add_trace(go.Scatter(
            x=xs[i], y=ys[i], mode="markers", name=club,
            marker=dict(size=14, color=club_colors[club], line=dict(color="white", width=1)),
            text=[f"Member {n}<br>Faction: {club}<br>Friends in club: {G.degree(n)}" for n in members],
            hoverinfo="text",
        ))
    # Hide the axes: positions come from the layout algorithm, so the numbers mean nothing
    fig.update_layout(
        title=title, width=750, height=650, plot_bgcolor="white",
        xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"),
    )
    return fig


fig = plot_network(G, layouts["Kamada-Kawai"], "Karate club friendships (Kamada-Kawai layout)")
fig.show()

Hover over any node to see who it is and how many friends they have; drag a box to zoom in on a crowded area, and double-click to zoom back out. Notice that we hid the axes entirely: since the positions came from a layout algorithm, tick marks and gridlines would suggest a scale that doesn't exist.

Because every layout is just a different set of positions for the same traces, we can also add a dropdown menu that swaps between them, so the reader can compare layouts for themselves. The dropdown uses Plotly's `"restyle"` method to replace the `x` and `y` data of each trace in place.

In [ ]:
fig = plot_network(G, layouts["Spring"], "Karate club friendships: choose a layout from the menu")

# TODO: build one dropdown button per layout, each replacing the x and y data of every trace
buttons = []
for name, pos in layouts.items():
    xs, ys = network_xy(G, pos)
    buttons.append(dict(label=name, method="restyle", args=[{"x": xs, "y": ys}]))

fig.update_layout(updatemenus=[dict(buttons=buttons, x=0, xanchor="left", y=1.08)])
fig.show()

Switch between layouts and watch the same nodes rearrange. Try finding member 0 (Mr. Hi) and member 33 (the president) in each layout: in which layouts is it obvious that they're the two central figures, and in which is it hidden?

**One limitation:** in Plotly you can't drag individual nodes around. The layout is fixed once it's computed, and the reader can only zoom and pan.

### An alternative: drag-and-drop networks with pyvis

The `pyvis` library takes a different approach to interactivity. Instead of computing a layout once and drawing it, pyvis runs a **live physics simulation** in the browser, similar to the spring layout but animated: nodes push each other apart, edges pull connected nodes together, and you can **grab any node and drag it** while the rest of the network reacts.

We build a pyvis `Network`, add each node with its color, size, and hover text, then add the edges. Notice that we never pick a layout: the physics simulation *is* the layout, and it runs again every time the chart loads.

The last line wraps pyvis's HTML in an `<iframe>` so the chart displays inside the notebook and survives the HTML export you submit. (pyvis's own `net.show()` writes a separate `.html` file, which wouldn't be included in your submission.)

In [ ]:
# !pip install pyvis

from pyvis.network import Network
from IPython.display import HTML
import html

# TODO: create a pyvis network, then add every node and edge from G
net = Network(height="600px", width="100%", bgcolor="white", cdn_resources="remote")

for n in G.nodes:
    club = G.nodes[n]["club"]
    net.add_node(
        n,
        label=str(n),
        color=club_colors[club],
        size=8 + 2 * G.degree(n),  # bigger nodes have more friends
        title=f"Member {n}\nFaction: {club}\nFriends in club: {G.degree(n)}",  # hover text
    )

for u, v in G.edges():
    net.add_edge(u, v, color="lightgray")

# Display the chart inside the notebook
HTML(f'<div><iframe srcdoc="{html.escape(net.generate_html())}" width="100%" height="620" style="border:none"></iframe></div>')

Watch the network settle into place when the cell runs, then try dragging member 0 or member 33 away from the group: their friends follow along, which is a very physical way to feel how connected they are. Hover over a node for its details, and scroll to zoom.

**Plotly or pyvis?**

- **pyvis** is better for *exploring* a network you don't know yet: dragging nodes apart is a fast, intuitive way to untangle a crowded area and see who is connected to whom.
- **Plotly** is better for *communicating* a finding: you choose the layout deliberately, the picture is the same every time someone opens it, and it fits alongside the other Plotly charts in a report or dashboard. With pyvis, the simulation can settle differently on each load, and readers may drag the network into a shape that no longer supports your point.

This is the same distinction between exploratory and explanatory visualization we've used all semester, applied to networks.

### Encoding importance: a scale-free network

So far every node has been drawn the same size, apart from the sizing exercise. In many networks, though, the most important thing to see is that **a few nodes are far more connected than everyone else**. Think of airline hubs, or a handful of accounts with millions of followers. Networks with this shape are called **scale-free**: most nodes have only a few connections, and a small number of **hubs** have a huge number.

`networkx` can generate one with `barabasi_albert_graph()`. It builds the network one node at a time, and each new node connects to existing nodes with a preference for the ones that are already well connected ("the rich get richer"). The number of connections a node has is called its **degree**.

To make the hubs stand out, we'll encode degree with **node size**. We'll also color by degree, as a redundant encoding just like `color="category"` in the treemap. One important detail: we set `sizemode="area"`, so a node's **area**, not its width, is proportional to its degree. Readers judge circles by area, so a node with twice the connections should take up twice the space, not four times.

In [ ]:
# TODO: generate a scale-free network with 150 nodes; each new node attaches to 2 existing nodes
SF = nx.barabasi_albert_graph(150, 2, seed=42)
sf_degree = dict(SF.degree())
sf_pos = nx.spring_layout(SF, seed=42)

edge_x, edge_y = [], []
for u, v in SF.edges():
    edge_x += [sf_pos[u][0], sf_pos[v][0], None]
    edge_y += [sf_pos[u][1], sf_pos[v][1], None]

# Draw the least connected nodes first, so the hubs sit on top
nodes = sorted(SF.nodes, key=lambda n: sf_degree[n])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode="lines",
    line=dict(color="lightgray", width=0.7),
    hoverinfo="skip", showlegend=False,
))
# TODO: size and color each node by its number of connections (degree)
fig.add_trace(go.Scatter(
    x=[sf_pos[n][0] for n in nodes],
    y=[sf_pos[n][1] for n in nodes],
    mode="markers",
    marker=dict(
        size=[sf_degree[n] for n in nodes],
        sizemode="area",                                   # area, not width, is proportional to degree
        sizeref=2 * max(sf_degree.values()) / (45 ** 2),   # the biggest hub is about 45 pixels across
        sizemin=3,                                         # keep the smallest nodes visible
        color=[sf_degree[n] for n in nodes],
        colorscale="Viridis",
        colorbar=dict(title="Connections"),
        line=dict(color="white", width=0.5),
    ),
    text=[f"Node {n}<br>Connections: {sf_degree[n]}" for n in nodes],
    hoverinfo="text", showlegend=False,
))
fig.update_layout(
    title="A scale-free network: node size shows number of connections",
    width=800, height=700, plot_bgcolor="white",
    xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"),
)
fig.show()

The hubs jump out immediately, and the spring layout has pulled them toward the center, since so many other nodes are connected to them. Hover over the biggest nodes to see just how many connections they have compared to the typical node at the edge.

A network diagram shows *that* there are hubs, but a plain histogram of degrees shows how lopsided the distribution really is:

In [ ]:
# TODO: plot a histogram of how many connections each node has
fig = px.histogram(
    x=list(sf_degree.values()),
    labels={"x": "Number of connections"},
    title="Most nodes have only a few connections; a handful of hubs have many",
)
fig.update_layout(yaxis_title="Number of nodes", bargap=0.1)
fig.show()

That long tail to the right is the signature of a scale-free network. Notice the two charts answer different questions: the histogram tells you *how many* hubs there are and how extreme they are, while the network diagram tells you *where* they sit and who they connect to. As with treemaps vs. sunbursts, pick the chart that matches your question, or pair them.


## On your own

### Part 1: trees

Using the same `sales` dataset, build a treemap or sunburst chart (your choice) with the hierarchy `region -> segment`, valued by any other metric instead of `sales`. In a markdown cell below, explain which chart type you chose and why it was the better fit for exploring *profit* by region and segment specifically, not just any hierarchy.

In [1]:
#TODO: Complete challenge here

**Type your Explanation of your design choices here**

### Part 2: networks

Using the karate club network `G`, build an interactive Plotly network chart using the **one layout** you think best communicates that the club split into two factions. Then make one additional encoding change that adds information, for example sizing each node by its number of friends (`G.degree(n)`), so the most connected members stand out.

In a markdown cell below, explain why you chose that layout, and why you rejected at least one of the other layouts.

In [ ]:
#TODO: Complete Part 2 here

**Type your explanation of your layout and encoding choices here**


## Submission

Export this notebook to HTML (`Jupyter: Export to HTML` from the Command Palette, or the "..." menu in the notebook toolbar) and submit the HTML file on Canvas.

## Where to go next: network visualization

We've only scratched the surface of network visualization here. It's a deep topic with its own analysis methods (finding communities, measuring how central each node is) and its own design problems, especially once a network grows past a few hundred nodes and every layout turns into a "hairball." If it interests you, some starting points:

- `networkx` documentation, including the full list of layout functions: <https://networkx.org/documentation/stable/reference/drawing.html#module-networkx.drawing.layout>
- Plotly's network graph tutorial: <https://plotly.com/python/network-graphs/>
- `pyvis`, for drag-and-drop interactive networks: <https://pyvis.readthedocs.io/>
- Gephi, a free desktop application for laying out and styling large networks: <https://gephi.org/>